In [ ]:
import os, subprocess
subprocess.run("pip install -q scikit-learn pandas statsmodels "numpy<2"", shell=True)
os.makedirs("/workspace/data", exist_ok=True)
os.makedirs("/workspace/outputs/nineseed", exist_ok=True)


# Nineseed — More seeds at the three shortest lengths (robustness of the flagship margin)

**Why this notebook exists.** Fragsweep measured the main fragmentation sweep on the CD-HIT-97% split at
3 seeds across 7 lengths, and an exact McNemar test on Fragsweep's own per-sequence predictions already
showed the linear SVM's advantage over the best CNN is statistically significant at every one of
those 3 seeds at 300/400/500/1000 bp (paper Table 3D) -- so this is not a notebook needed to prove
the effect exists. What it adds is seed *count*: reviewers specifically asked whether 3 seeds is
enough to trust the CNNs' short-fragment collapse as a real property of the architectures, rather
than 3 unlucky draws, given how large VGG-16's and Inception-v1's seed-to-seed standard deviations
already are in Table 3 (up to 0.117 and 0.261 respectively at some lengths). **This notebook adds 6
more seeds (seeds 3-8, for 9 total) at only the three lengths where the margin is largest and the
CNNs are least stable: 300, 400 and 500 bp.** It reuses the exact same CD-HIT-97% clustering, split
procedure, architectures, training loop and fragment-sampling code as Fragsweep (byte-identical
`fragment()` and `split_grouped()`), so every new number is directly comparable to Fragsweep's.

**What this buys, concretely.** With 9 seeds instead of 3 at these three lengths we can: (a) report
a tighter confidence interval on the margin itself; (b) estimate how *often* a CNN's training
collapses at short length, rather than only how much a mean is pulled down by one bad seed in 3;
and (c) run the McNemar test (already run on Fragsweep's 3 seeds in the paper's Table 3D) on 6 more
independent seeds, for a much larger effective sample when the per-seed p-values are combined.

**Runtime.** 6 extra seeds x 3 lengths x 3 CNNs = 54 CNN fits, the same per-fit cost as Fragsweep
(~3 min each on a T4), so roughly 2.5-3 hours of GPU time, plus a CPU-only SVM/RF pass at the same
seeds and lengths (a few minutes total). **This fits inside a 6-hour GPU budget with margin.** If
you are shorter on quota than that, two knobs below let you cut it down further:
- `EXTRA_SEEDS = [3,4,5]` (3 more seeds instead of 6) roughly halves the GPU time.
- `LENGTHS = [300, 500]` (drop 400 bp) cuts it by another third.

**Dataset.** All notebooks below read the *same* dataset you already used for the main experiments and the 5-model fragment sweep:

`/kaggle/input/datasets/yashrajrajarishi/subtype-classification/prepared_sequences_clean.csv`

with columns `sequence`, `subtype`, `cluster` (the MinHash cluster id). Attach that Kaggle Dataset to the notebook the same way you did for the fragment-test notebook (Add Data -> Your Datasets -> subtype-classification). **No new download is needed** for this file. Where a test needs something extra (a reference genome, an external tool), that is called out explicitly at the top of the notebook, with the exact cell that fetches or installs it -- it downloads automatically inside the notebook, you do not need to upload anything yourself.


**Kaggle session budget.** Kaggle notebook sessions cap out at **12 hours** and (for GPU) a **~30 GPU-hours/week** quota. Each notebook below states whether it needs a GPU and gives a rough runtime estimate at the default settings, plus a `FAST_MODE` toggle to shrink seeds/epochs/lengths if you are short on quota. If a run is approaching the 12 h wall, results are saved to `/kaggle/working` incrementally (per seed / per length) so a partial run is still usable -- just re-run with the remaining settings and concatenate the CSVs.


### 1. Imports and install CD-HIT (identical to Fragsweep)

In [ ]:

import warnings; warnings.filterwarnings("ignore")
import os, re, time, subprocess, shutil
import numpy as np, pandas as pd

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import f1_score, balanced_accuracy_score

OUT = "/workspace/outputs/nineseed"
os.makedirs(OUT, exist_ok=True)

if shutil.which("cd-hit-est") is None:
    print("cd-hit-est not found, installing...")
    subprocess.run("conda install -y -c bioconda cd-hit 2>/dev/null || true", shell=True)
    if shutil.which("cd-hit-est") is None:
        subprocess.run(
            "wget -q https://github.com/weizhongli/cdhit/releases/download/V4.8.1/cd-hit-v4.8.1-2019-0228.tar.gz "
            "&& tar xzf cd-hit-v4.8.1-2019-0228.tar.gz && cd cd-hit-v4.8.1-2019-0228 && make -j4 "
            "&& cp cd-hit-est /usr/local/bin/ && cp cd-hit /usr/local/bin/", shell=True)
print("cd-hit-est available:", shutil.which("cd-hit-est") is not None)

import tensorflow as tf
from tensorflow.keras.layers import (Input, Conv1D, BatchNormalization, ReLU, Add,
    GlobalAveragePooling1D, Dropout, Dense, MaxPooling1D,
    SpatialDropout1D, Layer, Concatenate, MaxPool1D)
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.utils import to_categorical



### 2. Config -- READ THIS before you Run All and walk away

In [ ]:

EXTRA_SEEDS = [3, 4, 5, 6, 7, 8]      # 6 more seeds beyond Fragsweep's [0,1,2] -- cut to [3,4,5] if short on GPU quota
LENGTHS     = [300, 400, 500]         # the three lengths this notebook targets -- drop 400 to save time if needed
DEEP_MODELS = ["resnet18", "vgg16", "inception"]
MAX_TFIDF_FEATS = 512
IDENTITY = 0.97   # CD-HIT-97%, the split this paper treats as primary (same as Fragsweep)
EPOCHS = 120     # matches the original per-architecture notebooks (was 60 -- bug, now fixed)
BATCH  = 48


### 3. Load data and write FASTA for CD-HIT (identical to Fragsweep -- same FASTA, same clustering)

In [ ]:

DATA_PATH = "/workspace/data/prepared_sequences_clean.csv"
df = pd.read_csv(DATA_PATH).reset_index(drop=True)

le = LabelEncoder(); y_all = le.fit_transform(df["subtype"].values)
CLASSES = list(le.classes_); n_cls = len(CLASSES)
X_all = df["sequence"].values
n = len(df)
print(f"{n} sequences, {n_cls} classes: {CLASSES}")

with open(f"{OUT}/pol_nineseed.fasta", "w") as f:
    for i, s in enumerate(X_all):
        f.write(f">{i}\n{s}\n")


### 4. Run CD-HIT at 97% identity (same command, same word size as Fragsweep -- must reproduce the same 7,182 clusters for these results to be comparable to Table 3)

In [ ]:

def run_cdhit(identity, word_size):
    out = f"{OUT}/pol_nineseed_cdhit_{int(identity*100)}"
    cmd = f"cd-hit-est -i {OUT}/pol_nineseed.fasta -o {out} -c {identity} -n {word_size} -M 16000 -T 4 -d 0"
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return out, r.returncode

def parse_cdhit_clusters(clstr_path, n_seqs):
    labels = np.full(n_seqs, -1, dtype=int)
    cid = -1
    with open(clstr_path) as f:
        for line in f:
            if line.startswith(">Cluster"):
                cid += 1
            else:
                idx = int(line.split(">")[1].split("...")[0])
                labels[idx] = cid
    assert (labels >= 0).all()
    return labels

out, rc = run_cdhit(IDENTITY, 10)
assert rc == 0 and os.path.exists(out + ".clstr"), f"CD-HIT failed (returncode={rc})"
groups = parse_cdhit_clusters(out + ".clstr", n)
n_clusters = len(set(groups))
print(f"CD-HIT 97% identity: {n_clusters} clusters "
      f"(Fragsweep reported 7,182 -- this should match; if it doesn't, stop and check the CD-HIT version/args)")


### 5. Fragmentation + split helpers -- byte-identical to Fragsweep's, so seeds 0-8 form one consistent series

In [ ]:

def fragment(seq, L, rng):
    if L is None or len(seq) <= L:
        return seq
    start = rng.integers(0, len(seq) - L + 1)   # same upper bound as Fragsweep (gen_fragsweep.py); NOT the CdhitBaseline convention
    return seq[start:start+L]

def split_grouped(X, y, grp, seed):
    s1 = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=seed)
    trv, te = next(s1.split(X, y, grp))
    s2 = StratifiedGroupKFold(n_splits=4, shuffle=True, random_state=seed)
    tr_rel, va_rel = next(s2.split(X[trv], y[trv], grp[trv]))
    return trv[tr_rel], trv[va_rel], te

def fit_tfidf(seq_tr, seq_va, seq_te):
    vec = TfidfVectorizer(analyzer="char", ngram_range=(6,6), max_features=MAX_TFIDF_FEATS, lowercase=False)
    Xtr = vec.fit_transform(seq_tr); Xva = vec.transform(seq_va); Xte = vec.transform(seq_te)
    return Xtr, Xva, Xte


### 6. SVM + RF at the 6 extra seeds (cheap, CPU only -- for a consistent, uniform-N supplementary table)

In [ ]:

rows = []
pred_rows = []
splits_cache = {}

for L in LENGTHS:
    for seed in EXTRA_SEEDS:
        rng = np.random.default_rng(seed)
        seqs = np.array([fragment(s, L, rng) for s in X_all])
        tr, va, te = split_grouped(seqs, y_all, groups, seed)
        splits_cache[(L, seed)] = (tr, va, te)

        Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
        for model_name, clf in [
            ("svm", LinearSVC(class_weight="balanced", random_state=seed, max_iter=20000)),
            ("rf", RandomForestClassifier(n_estimators=400, class_weight="balanced", random_state=seed, n_jobs=-1)),
        ]:
            clf.fit(Xtr, y_all[tr])
            pred = clf.predict(Xte)
            rows.append(dict(length=L, seed=seed, model=model_name,
                              f1=f1_score(y_all[te], pred, average="macro"),
                              bal_acc=balanced_accuracy_score(y_all[te], pred)))
            pred_rows.extend([dict(length=L, seed=seed, model=model_name,
                                    test_idx=int(i), y_true=int(t), y_pred=int(p))
                               for i, t, p in zip(te, y_all[te], pred)])
    pd.DataFrame(rows).to_csv(f"{OUT}/nineseed_classical_raw.csv", index=False)
    pd.DataFrame(pred_rows).to_csv(f"{OUT}/nineseed_classical_predictions.csv", index=False)
    print(f"[classical] length={L} done, all {len(EXTRA_SEEDS)} extra seeds")


### 7. CNNs at the 6 extra seeds x 3 lengths -- needs GPU. Same architectures and training loop as Fragsweep.

In [ ]:


class StochasticDepth(Layer):
    def __init__(self, p=0.85, **kw): super().__init__(**kw); self.p = p
    def call(self, x, training=None):
        if not training: return x
        r = tf.random.uniform([tf.shape(x)[0], 1, 1])
        return x * tf.floor(self.p + r) / self.p
    def get_config(self): return {**super().get_config(), "p": self.p}

def res_block(x, f, stride=1, sp=0.85, dr=0.1):
    sc = x
    x = Conv1D(f, 3, strides=stride, padding="same", use_bias=False)(x)
    x = ReLU()(BatchNormalization()(x)); x = SpatialDropout1D(dr)(x)
    x = BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x))
    if stride != 1 or sc.shape[-1] != f:
        sc = BatchNormalization()(Conv1D(f, 1, strides=stride, padding="same", use_bias=False)(sc))
    return ReLU()(Add()([StochasticDepth(sp)(x), sc]))

def build_resnet18(input_len, n_classes):
    inp = Input((input_len, 1)); x = inp
    x = ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same", use_bias=False)(x)))
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    for f, sp in [(32, .95), (32, .92)]:  x = res_block(x, f, 1, sp)
    for f, sp in [(64, .88), (64, .85)]:  x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(128, .80), (128, .78)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    for f, sp in [(256, .75), (256, .72)]: x = res_block(x, f, 2 if f != x.shape[-1] else 1, sp)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def build_vgg16(input_len, n_classes):
    def conv_block(x, f, n_convs):
        for _ in range(n_convs):
            x = ReLU()(BatchNormalization()(Conv1D(f, 3, padding="same", use_bias=False)(x)))
        return MaxPooling1D(2, padding="same")(x)
    inp = Input((input_len, 1))
    x = conv_block(inp, 32, 2)
    x = conv_block(x,   64, 2)
    x = conv_block(x,  128, 3)
    x = conv_block(x,  256, 3)
    x = conv_block(x,  256, 3)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    x = Dropout(0.4)(ReLU()(Dense(256)(x)))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

def inception_module(x, f1, f3r, f3, f5r, f5, fp):
    b1 = Conv1D(f1, 1, padding="same", activation="relu")(x)
    b3 = Conv1D(f3, 3, padding="same", activation="relu")(Conv1D(f3r, 1, padding="same", activation="relu")(x))
    b5 = Conv1D(f5, 5, padding="same", activation="relu")(Conv1D(f5r, 1, padding="same", activation="relu")(x))
    bp = Conv1D(fp, 1, padding="same", activation="relu")(MaxPooling1D(3, strides=1, padding="same")(x))
    return Concatenate()([b1, b3, b5, bp])

def build_inception(input_len, n_classes):
    inp = Input((input_len, 1))
    x = MaxPooling1D(3, strides=2, padding="same")(
            ReLU()(BatchNormalization()(Conv1D(32, 7, strides=2, padding="same")(inp))))
    x = inception_module(x, 16, 24, 32, 4,  8,  8)
    x = inception_module(x, 32, 32, 48, 8, 24, 16)
    x = MaxPooling1D(3, strides=2, padding="same")(x)
    x = inception_module(x, 48, 24, 52, 8, 24, 16)
    x = Dropout(0.4)(GlobalAveragePooling1D()(x))
    out = Dense(n_classes, activation="softmax")(x)
    m = Model(inp, out)
    m.compile(optimizer=Adam(3e-4), loss=CategoricalCrossentropy(label_smoothing=0.1), metrics=["accuracy"])
    return m

BUILDERS = {"resnet18": build_resnet18, "vgg16": build_vgg16, "inception": build_inception}
DEEP_MODELS = ["resnet18", "vgg16", "inception"]

def run_deep_original(model_name, Xtr, ytr, Xva, yva, Xte, yte, seed, n_classes):
    """Exact training recipe from the original per-architecture notebooks (verified byte-identical
    in Phase A, same code already used in Fragsweep/Ambig/Enc): per-feature StandardScaler (fit on train
    only), one-hot labels, CategoricalCrossentropy(label_smoothing=0.1), Adam(3e-4), up to 120
    epochs, EarlyStopping(patience=15) + ReduceLROnPlateau(factor=0.5, patience=6, min_lr=1e-6)."""
    tf.random.set_seed(seed)
    sc = StandardScaler()
    Xtr_d = sc.fit_transform(np.asarray(Xtr.todense() if hasattr(Xtr, "todense") else Xtr))[..., None]
    Xva_d = sc.transform(np.asarray(Xva.todense() if hasattr(Xva, "todense") else Xva))[..., None]
    Xte_d = sc.transform(np.asarray(Xte.todense() if hasattr(Xte, "todense") else Xte))[..., None]
    cw = dict(enumerate(compute_class_weight("balanced", classes=np.unique(ytr), y=ytr)))
    model = BUILDERS[model_name](Xtr_d.shape[1], n_classes)
    model.fit(Xtr_d, to_categorical(ytr, n_classes),
              validation_data=(Xva_d, to_categorical(yva, n_classes)),
              epochs=EPOCHS, batch_size=BATCH, class_weight=cw, shuffle=True, verbose=0,
              callbacks=[EarlyStopping("val_loss", patience=15, restore_best_weights=True),
                         ReduceLROnPlateau("val_loss", factor=0.5, patience=6, min_lr=1e-6)])
    prob = model.predict(Xte_d, verbose=0)
    pred = prob.argmax(1)
    f1 = f1_score(yte, pred, average="macro"); bal = balanced_accuracy_score(yte, pred)
    tf.keras.backend.clear_session()
    return f1, bal, pred

DEEP_RAW_PATH = f"{OUT}/nineseed_deep_raw.csv"
if os.path.exists(DEEP_RAW_PATH):
    _existing_deep = pd.read_csv(DEEP_RAW_PATH)
    deep_done_keys = set(zip(_existing_deep["length"], _existing_deep["seed"], _existing_deep["model"]))
    deep_rows = _existing_deep.to_dict("records")
    print(f"Resuming deep models: {len(deep_rows)} rows already done.")
else:
    deep_rows = []
    deep_done_keys = set()

deep_pred_rows = []
for (L, seed), (tr, va, te) in splits_cache.items():
    remaining = [m for m in DEEP_MODELS if (L, seed, m) not in deep_done_keys]
    if not remaining:
        continue
    rng = np.random.default_rng(seed)
    seqs = np.array([fragment(s, L, rng) for s in X_all])
    Xtr, Xva, Xte = fit_tfidf(seqs[tr], seqs[va], seqs[te])
    for m in remaining:
        f1, bal, pred = run_deep_original(m, Xtr, y_all[tr], Xva, y_all[va], Xte, y_all[te], seed, n_cls)
        deep_rows.append(dict(length=L, seed=seed, model=m, f1=f1, bal_acc=bal))
        deep_pred_rows.extend([dict(length=L, seed=seed, model=m,
                                     test_idx=int(i), y_true=int(t), y_pred=int(p))
                                for i, t, p in zip(te, y_all[te], pred)])
        pd.DataFrame(deep_rows).to_csv(DEEP_RAW_PATH, index=False)   # incremental save
        print(f"[deep] length={L} seed={seed} model={m} f1={f1:.4f} done")
    pd.DataFrame(deep_pred_rows).to_csv(f"{OUT}/nineseed_deep_predictions.csv", index=False)

nineseed_all = pd.concat([pd.DataFrame(rows), pd.DataFrame(deep_rows)], ignore_index=True)
nineseed_all_predictions = pd.concat([pd.DataFrame(pred_rows), pd.DataFrame(deep_pred_rows)], ignore_index=True)
nineseed_all.to_csv(f"{OUT}/nineseed_all_raw.csv", index=False)
nineseed_all_predictions.to_csv(f"{OUT}/nineseed_all_predictions.csv", index=False)


### 8. Combine with Fragsweep's original 3 seeds for a 9-seed summary at these three lengths

**Edit the path below to point at your uploaded Fragsweep outputs** (`fragsweep_all_raw.csv` /  `fragsweep_all_predictions.csv`, from `fragsweep_fragmentation_rerun_outputs.zip`) -- attach that as a Kaggle Dataset the same way you did for this notebook's main dataset, so the combination step below can run without you pasting numbers in by hand.

In [ ]:

Fragsweep_DIR = "/workspace/data/fragsweep_outputs"

fragsweep_raw_path = f"{Fragsweep_DIR}/fragsweep_all_raw.csv"
fragsweep_pred_path = f"{Fragsweep_DIR}/fragsweep_all_predictions.csv"

if os.path.exists(fragsweep_raw_path):
    fragsweep_raw = pd.read_csv(fragsweep_raw_path)
    fragsweep_raw = fragsweep_raw[(fragsweep_raw["identity"] == 0.97) & (fragsweep_raw["length"].astype(str).isin(["300","400","500"]))]
    fragsweep_raw["length"] = fragsweep_raw["length"].astype(int)
    combined = pd.concat([fragsweep_raw[["length","seed","model","f1","bal_acc"]], nineseed_all], ignore_index=True)
    combined.to_csv(f"{OUT}/nineseed_combined_9seed_raw.csv", index=False)
    print("9-seed summary (mean +/- std macro-F1), 300/400/500 bp:")
    print(combined.groupby(["length","model"])["f1"].agg(["mean","std","count"]).round(4))
else:
    print(f"Fragsweep outputs not found at {fragsweep_raw_path} -- attach the Fragsweep outputs dataset to combine seed "
          "counts. nineseed_all_raw.csv (this notebook's own 6 seeds) is still saved and usable on its own.")


In [ ]:
!cd /workspace/outputs/nineseed && zip -r nineseed_extra_seeds_outputs.zip nineseed_all_raw.csv nineseed_all_predictions.csv nineseed_classical_raw.csv nineseed_classical_predictions.csv nineseed_deep_raw.csv nineseed_deep_predictions.csv nineseed_combined_9seed_raw.csv 2>/dev/null; ls -la /workspace/outputs/nineseed/*.zip